# HFT hw2 — PEPE EDA (USDT target + USDC leader)

Sample days: train `2025-11-01…11-07`, val `2026-08-10…08-15`.

Pipeline: unpack → `scripts/run_pepe_eda.py` → this notebook reads `outputs/pepe_eda_results.json`.

Reports: `reports/VARIABLE_ANALYSIS_LOG.md`, `reports/EXPLANATIONS.md`, `CONTEXT.md`.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

from hft_common import TRAIN_DAYS, VAL_DAYS, ALL_DAYS, HORIZONS, load_day_pair

RESULTS = ROOT / 'outputs' / 'pepe_eda_results.json'
assert RESULTS.exists(), 'Run: python scripts/run_pepe_eda.py'
res = json.loads(RESULTS.read_text())
res.keys()

## V01–V02 Data quality and targets

Book almost fills the 172 800 grid but starts a few seconds late. Targets have many exact zeros on short horizons.

In [ ]:
dq = pd.json_normalize(res['dq'])
cols = [
    'day',
    '1000PEPEUSDT_PERP:book.n',
    '1000PEPEUSDT_PERP:book.first_lag_ms',
    '1000PEPEUSDT_PERP:book.max_gap_ms',
    '1000PEPEUSDT_PERP:book.locked_frac',
    '1000PEPEUSDT_PERP:targets.y_500ms_zero_frac',
    '1000PEPEUSDT_PERP:targets.y_2s_zero_frac',
    '1000PEPEUSDT_PERP:targets.y_8s_zero_frac',
    '1000PEPEUSDT_PERP:targets.y_2s_std_bp',
]
dq[cols].round(4)

## V03 Own imbalance → y

Spearman IC by day/horizon, plus S-shape of mean `y_500ms` vs imb bins (train).

In [ ]:
print('train IC', res['imb_ic']['train']['mean_spearman'])
print('val IC', res['imb_ic']['val']['mean_spearman'])
print('|imb|>0.9 frac train/val',
      res['imb_ic']['train']['mean_ext_frac'],
      res['imb_ic']['val']['mean_ext_frac'])

sshape = res['imb_ic']['train']['sshape_y500_bp']
keys = list(sshape.keys())
vals = [sshape[k] for k in keys]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(len(vals)), vals, marker='o')
ax.axhline(0, color='grey', lw=0.8)
ax.set_title('Train S-shape: mean y_500ms (bp) by imb bin')
ax.set_xlabel('imb bin (−1 → +1)')
ax.set_ylabel('bp')
plt.tight_layout()
plt.show()

## V04 Lead–lag PEPEUSDC → PEPEUSDT (100 ms)

Median cross-correlation: `corr(r_USDC(t−k), r_USDT(t))`. Positive k means USDC moved first.

In [ ]:
tr = res['leadlag']['train_median_100ms']
va = res['leadlag']['val_median_100ms']
tr5 = res['leadlag']['train_median_500ms']
va5 = res['leadlag']['val_median_500ms']
print('100ms train peak k>0', tr['peak_lag_ms'], 'ms', tr['peak_corr'], 'rev@100', tr['rev_corr_100'])
print('100ms val peak k>0', va['peak_lag_ms'], 'ms', va['peak_corr'], 'rev@100', va['rev_corr_100'])
print('500ms train/val peak', tr5['peak_lag_ms'], tr5['peak_corr'], va5['peak_lag_ms'], va5['peak_corr'])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(tr['lag_ms'], tr['median_curve'], label='train USDC→USDT')
axes[0].plot(va['lag_ms'], va['median_curve'], label='val USDC→USDT')
axes[0].plot(tr['lag_ms'], tr['median_reverse'], '--', label='train reverse')
axes[0].axhline(0, color='grey', lw=0.8)
axes[0].set_title('100 ms grid')
axes[0].set_xlabel('lag ms')
axes[0].legend(fontsize=8)
axes[1].plot(tr5['lag_ms'], tr5['median_curve'], label='train USDC→USDT')
axes[1].plot(va5['lag_ms'], va5['median_curve'], label='val USDC→USDT')
axes[1].plot(tr5['lag_ms'], tr5['median_reverse'], '--', label='train reverse')
axes[1].axhline(0, color='grey', lw=0.8)
axes[1].set_title('500 ms grid')
axes[1].set_xlabel('lag ms')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## V05 Gap vs USDC return

Same-coin `gap_dev` vs last-step USDC mid return as predictors of `y`.

In [ ]:
print('train', res['gap_ic']['train'])
print('val', res['gap_ic']['val'])

## V06 PEPE-only SCORE — imbalance baseline + M1

Not the full 12-cell competition SCORE (needs BTC/ETH/SOL). Mean of 3 PEPE horizons after train-clip + daily Pearson.

In [ ]:
base = res['baseline']
print('M0 train', base['train'])
print('M0 val', base['val'])
print('M1 val', base['m1_val'])
pd.DataFrame(base['val_daily'])

## Optional: reload one day

Quick look at aligned USDT+USDC frame.

In [ ]:
day = TRAIN_DAYS[0]
df = load_day_pair(day)
df[['ts', 'mid', 'imb', 'mid_usdc', 'gap', 'y_500ms', 'y_2s', 'y_8s']].head(10)